Will be using Vision Capable LLM to read and interpret the images, Instead of using OCR

In [ ]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

In [ ]:
# Encode the image to base64 and send it to LLM

with open("blood_work.png", "rb") as image_file:
    encoded_image = base64.b64encode(image_file.read()).decode("utf-8")


In [ ]:
llm = ChatGroq(model="qwen/qwen3.6-27b", temperature=0)

message = HumanMessage(content=[
    {"type" :"image_url","image_url":{"url":f"data:image/png;base64,{encoded_image}"}},
    {"type":"text","text":"Please analyze the blood work report and provide a summary of the results."}
])

response = llm.invoke([message])
print(response.content[0]["text"])

In [ ]:
# using the response from LLM we can create a agent to recommend diet plans
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [ ]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [ ]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)